# Basetable Ultimate

One clean daily row per date (Jul 5 – Nov 4 2024).  
Target: `polymarket_trump_prob`  
Sources: Polymarket · Time · Bluesky · Reddit · Newspapers · Google Trends · Polls · Financials

| Section | Features |
|---|---|
| 1. Target | Polymarket Trump win probability |
| 2. Time Dimension | Days to election, weekend dummy, weekday, campaign week |
| 3. Bluesky | Volume/engagement + RoBERTa sentiment + NRCLex emotions per buzz group |
| 4. Reddit | Volume/engagement + RoBERTa sentiment + NRCLex emotions per buzz group |
| 5. Newspapers | Volume per leaning + VADER sentiment + NRCLex emotions per leaning |
| 6. Google Trends | Daily search interest for 12 topic keywords |
| 7. Polls | Daily aggregated poll numbers + 7-day rolling avg |
| 8. Financials | S&P500, Oil, VIX, bonds, USD + derived returns |
| 9. Merge | Left join everything on date |
| 10. Lags | Polymarket lags 1–5, price change lags 1–3, key signal lags |
| 11. Save | `Data/3_Gold/basetable_ultimate.csv` |

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("../Functions").resolve()))
from buzz_column import add_buzz_labels

DATE_MIN = pd.Timestamp("2024-07-05")
DATE_MAX = pd.Timestamp("2024-11-04")
DATES    = pd.date_range(DATE_MIN, DATE_MAX, freq="D")

EMOTION_COLS = ["fear", "anger", "anticipation", "trust", "surprise", "sadness", "disgust", "joy"]

print(f"Date window : {DATE_MIN.date()} → {DATE_MAX.date()}  ({len(DATES)} days)")

Date window : 2024-07-05 → 2024-11-04  (123 days)


---
## 1. Target — Polymarket

In [2]:
# ─── Load Polymarket win probabilities as the base of the basetable ───
polymarket = pd.read_csv("../data/1_bronze/polymarket/polymarket_win_probabilities.csv", parse_dates=["date"])

# The Polymarket scrape runs at ~00:03 UTC each day.
# A scrape on 2024-07-14 00:03 captures the market price at the END of 2024-07-13.
# We therefore subtract 1 day from the scraped date so the price is correctly
# aligned with the calendar day it represents.
# Example: raw date 2024-07-14 → relabeled 2024-07-13 (end-of-day price for July 13).
polymarket["date"] = polymarket["date"] - pd.Timedelta(days=1)

# Keep only Trump probability — this is the dependent variable
basetable = polymarket[["date", "Trump (%)"]].rename(columns={"Trump (%)": "polymarket_trump_prob"})
basetable["polymarket_trump_prob"] = basetable["polymarket_trump_prob"] / 100  # scale to [0, 1]

basetable = basetable.sort_values("date").reset_index(drop=True)

# Lag-1: yesterday's probability (autoregressive feature)
basetable["polymarket_trump_prob_lag1"] = basetable["polymarket_trump_prob"].shift(1)

print(f"Base: {len(basetable)} rows from {basetable['date'].min().date()} to {basetable['date'].max().date()}")
basetable.head()

Base: 128 rows from 2024-06-30 to 2024-11-04


,date,polymarket_trump_prob,polymarket_trump_prob_lag1
0,2024-06-30,0.660,NaN
1,2024-07-01,0.650,0.660
2,2024-07-02,0.615,0.650
3,2024-07-03,0.605,0.615
4,2024-07-04,0.605,0.605


---
## 2. Time Dimension

In [3]:
ELECTION_DAY = pd.Timestamp("2024-11-05")

time_features = pd.DataFrame({"date": DATES})

time_features["days_to_election"] = (ELECTION_DAY - time_features["date"]).dt.days
time_features["is_weekend"]       = time_features["date"].dt.dayofweek.isin([5, 6]).astype(int)
time_features["weekday"]          = time_features["date"].dt.dayofweek   # 0=Mon … 6=Sun
time_features["campaign_week"]    = ((time_features["date"] - DATE_MIN).dt.days // 7) + 1

print(f"Time features: {len(time_features)} rows, {len(time_features.columns)} cols")
print(time_features.columns.tolist())
time_features.head(7)

Time features: 123 rows, 5 cols
['date', 'days_to_election', 'is_weekend', 'weekday', 'campaign_week']


,date,days_to_election,is_weekend,weekday,campaign_week
0,2024-07-05,123,0,4,1
1,2024-07-06,122,1,5,1
2,2024-07-07,121,1,6,1
3,2024-07-08,120,0,0,1
4,2024-07-09,119,0,1,1
5,2024-07-10,118,0,2,1
6,2024-07-11,117,0,3,1


---
## 3. Bluesky

In [5]:
# ─── Bluesky: volume & engagement ───
bsky_raw = pd.read_csv(
    "../data/2_silver/bluesky/bluesky_clean.csv",
    usecols=["timestamp", "candidate", "likes", "reposts", "replies", "is_reply"]
)
bsky_raw["date"] = (
    pd.to_datetime(bsky_raw["timestamp"], format="ISO8601", utc=True)
    .dt.normalize().dt.tz_localize(None)
)
bsky_raw = bsky_raw[(bsky_raw["date"] >= DATE_MIN) & (bsky_raw["date"] <= DATE_MAX)]

bsky_vol = bsky_raw.groupby("date").agg(
    bsky_total_posts  = ("candidate", "count"),
    bsky_trump_posts  = ("candidate", lambda x: (x == "TrumpBuzz").sum()),
    bsky_harris_posts = ("candidate", lambda x: (x == "HarrisBuzz").sum()),
    bsky_avg_likes    = ("likes",     "mean"),
    bsky_avg_reposts  = ("reposts",   "mean"),
    bsky_reply_ratio  = ("is_reply",  "mean"),
).reset_index()

bsky_vol["bsky_trump_share"]  = bsky_vol["bsky_trump_posts"]  / bsky_vol["bsky_total_posts"]
bsky_vol["bsky_harris_share"] = bsky_vol["bsky_harris_posts"] / bsky_vol["bsky_total_posts"]
bsky_vol = bsky_vol.drop(columns=["bsky_trump_posts", "bsky_harris_posts"])

print(f"Bluesky volume: {len(bsky_vol)} rows, {len(bsky_vol.columns)} cols")
print(bsky_vol.columns.tolist())

Bluesky volume: 123 rows, 7 cols
['date', 'bsky_total_posts', 'bsky_avg_likes', 'bsky_avg_reposts', 'bsky_reply_ratio', 'bsky_trump_share', 'bsky_harris_share']


In [10]:
bsky_vol.head()

,date,bsky_total_posts,bsky_avg_likes,bsky_avg_reposts,bsky_reply_ratio,bsky_trump_share,bsky_harris_share
0,2024-07-05,41,1.268293,0.439024,0.439024,0.341463,0.341463
1,2024-07-06,24,1.208333,0.500000,0.125000,0.416667,0.500000
2,2024-07-07,29,1.758621,0.482759,0.275862,0.448276,0.413793
3,2024-07-08,25,2.840000,1.280000,0.040000,0.480000,0.400000
4,2024-07-09,39,1.897436,0.589744,0.179487,0.410256,0.435897


In [7]:
# ─── Bluesky: RoBERTa sentiment + NRCLex emotions per buzz group ───
bsky_sent = pd.read_csv("../data/2_silver/bluesky/sentiment_bluesky.csv")
bsky_sent["date"] = pd.to_datetime(bsky_sent["date"], utc=True).dt.normalize().dt.tz_localize(None)
bsky_sent = bsky_sent[(bsky_sent["date"] >= DATE_MIN) & (bsky_sent["date"] <= DATE_MAX)]

frames = []
for group, prefix in [("TrumpBuzz", "bsky_trump"), ("HarrisBuzz", "bsky_harris"), ("ElectionBuzz", "bsky_election")]:
    sub = bsky_sent[bsky_sent["buzz_group"] == group].groupby("date").agg(
        sent_avg  = ("roberta_sentiment", "mean"),
        sent_std  = ("roberta_sentiment", "std"),
        pos_share = ("roberta_pos",       "mean"),
        neg_share = ("roberta_neg",       "mean"),
        **{e: (e, "mean") for e in EMOTION_COLS}
    )
    sub.columns = [f"{prefix}_{c}" for c in sub.columns]
    frames.append(sub)

bsky_sentiment = pd.concat(frames, axis=1).reset_index()
bsky_sentiment["bsky_sentiment_gap"] = (
    bsky_sentiment["bsky_trump_sent_avg"] - bsky_sentiment["bsky_harris_sent_avg"]
)

print(f"Bluesky sentiment: {len(bsky_sentiment)} rows, {len(bsky_sentiment.columns)} cols")
print(bsky_sentiment.columns.tolist())

Bluesky sentiment: 123 rows, 38 cols
['date', 'bsky_trump_sent_avg', 'bsky_trump_sent_std', 'bsky_trump_pos_share', 'bsky_trump_neg_share', 'bsky_trump_fear', 'bsky_trump_anger', 'bsky_trump_anticipation', 'bsky_trump_trust', 'bsky_trump_surprise', 'bsky_trump_sadness', 'bsky_trump_disgust', 'bsky_trump_joy', 'bsky_harris_sent_avg', 'bsky_harris_sent_std', 'bsky_harris_pos_share', 'bsky_harris_neg_share', 'bsky_harris_fear', 'bsky_harris_anger', 'bsky_harris_anticipation', 'bsky_harris_trust', 'bsky_harris_surprise', 'bsky_harris_sadness', 'bsky_harris_disgust', 'bsky_harris_joy', 'bsky_election_sent_avg', 'bsky_election_sent_std', 'bsky_election_pos_share', 'bsky_election_neg_share', 'bsky_election_fear', 'bsky_election_anger', 'bsky_election_anticipation', 'bsky_election_trust', 'bsky_election_surprise', 'bsky_election_sadness', 'bsky_election_disgust', 'bsky_election_joy', 'bsky_sentiment_gap']


In [8]:
bsky_sentiment.head()

,date,bsky_trump_sent_avg,bsky_trump_sent_std,bsky_trump_pos_share,bsky_trump_neg_share,bsky_trump_fear,bsky_trump_anger,bsky_trump_anticipation,bsky_trump_trust,bsky_trump_surprise,...,bsky_election_neg_share,bsky_election_fear,bsky_election_anger,bsky_election_anticipation,bsky_election_trust,bsky_election_surprise,bsky_election_sadness,bsky_election_disgust,bsky_election_joy,bsky_sentiment_gap
0,2024-07-05,-0.482043,0.440737,0.094004,0.576047,0.073353,0.113733,0.018786,0.082349,0.128771,...,0.330713,0.028083,0.012821,0.045564,0.128593,0.038462,0.016601,0.012327,0.036137,-0.547041
1,2024-07-06,-0.372417,0.468239,0.119959,0.492377,0.109973,0.105856,0.054973,0.059314,0.362371,...,0.135040,0.050000,0.100000,0.150000,0.150000,0.050000,0.050000,0.050000,0.150000,-0.236525
2,2024-07-07,-0.541296,0.384457,0.076313,0.617609,0.051515,0.059614,0.024416,0.072629,0.389185,...,0.413623,0.000000,0.041667,0.041667,0.166667,0.041667,0.041667,0.000000,0.000000,-0.549087
3,2024-07-08,-0.415200,0.614197,0.167201,0.582401,0.076709,0.095620,0.066667,0.061966,0.040278,...,0.109070,0.066667,0.077778,0.055556,0.033333,0.066667,0.055556,0.044444,0.033333,-0.340401
4,2024-07-09,-0.317252,0.525823,0.132292,0.449544,0.046917,0.058850,0.049699,0.152253,0.166585,...,0.570145,0.000000,0.041667,0.033333,0.080952,0.108333,0.000000,0.083333,0.000000,-0.398980


---
## 4. Reddit

In [ ]:
# ─── Reddit: volume & engagement ───
posts = pd.read_parquet(
    "../data/2_silver/reddit/reddit_posts_clean.parquet",
    columns=["created_utc", "candidate", "subreddit", "score", "upvote_ratio", "num_comments"]
)
posts["date"] = (
    pd.to_datetime(posts["created_utc"], utc=True)
    .dt.normalize().dt.tz_localize(None)
)
posts = posts[(posts["date"] >= DATE_MIN) & (posts["date"] <= DATE_MAX)]

reddit_vol = posts.groupby("date").agg(
    reddit_total_posts        = ("candidate",    "count"),
    reddit_trump_posts        = ("candidate",    lambda x: (x == "TrumpBuzz").sum()),
    reddit_harris_posts       = ("candidate",    lambda x: (x == "HarrisBuzz").sum()),
    reddit_avg_score          = ("score",        "mean"), # net vote = upvotes - downvotes
    reddit_avg_upvote_ratio   = ("upvote_ratio", "mean"),
    reddit_avg_comments       = ("num_comments", "mean"),
    reddit_conservative_posts = ("subreddit",    lambda x: x.isin(["conservative", "trump"]).sum()),
    reddit_liberal_posts      = ("subreddit",    lambda x: x.isin(["liberal", "democrats"]).sum()),
).reset_index()

reddit_vol["reddit_trump_share"]        = reddit_vol["reddit_trump_posts"]        / reddit_vol["reddit_total_posts"]
reddit_vol["reddit_harris_share"]       = reddit_vol["reddit_harris_posts"]       / reddit_vol["reddit_total_posts"]
reddit_vol["reddit_conservative_share"] = reddit_vol["reddit_conservative_posts"] / reddit_vol["reddit_total_posts"]
reddit_vol["reddit_liberal_share"]      = reddit_vol["reddit_liberal_posts"]      / reddit_vol["reddit_total_posts"]
reddit_vol = reddit_vol.drop(columns=["reddit_trump_posts", "reddit_harris_posts",
                                       "reddit_conservative_posts", "reddit_liberal_posts"])

print(f"Reddit volume: {len(reddit_vol)} rows, {len(reddit_vol.columns)} cols")
print(reddit_vol.columns.tolist())

Reddit volume: 122 rows, 9 cols
['date', 'reddit_total_posts', 'reddit_avg_score', 'reddit_avg_upvote_ratio', 'reddit_avg_comments', 'reddit_trump_share', 'reddit_harris_share', 'reddit_conservative_share', 'reddit_liberal_share']


In [12]:
reddit_vol.head()

,date,reddit_total_posts,reddit_avg_score,reddit_avg_upvote_ratio,reddit_avg_comments,reddit_trump_share,reddit_harris_share,reddit_conservative_share,reddit_liberal_share
0,2024-07-05,726,310.015152,0.805399,72.249311,0.275482,0.509642,0.241047,0.144628
1,2024-07-06,635,356.856693,0.805417,77.522835,0.218898,0.527559,0.253543,0.121260
2,2024-07-07,565,325.943363,0.817752,61.196460,0.233628,0.536283,0.263717,0.141593
3,2024-07-08,668,322.062874,0.815569,85.366766,0.252994,0.502994,0.248503,0.166168
4,2024-07-09,718,358.575209,0.809624,80.502786,0.338440,0.448468,0.233983,0.144847


In [14]:
# ─── Reddit: VADER sentiment + NRCLex emotions per buzz group ───
# Requires 1.5_sentiment_analysis.ipynb to have been run first
REDDIT_SENT_PATH = "../data/2_silver/reddit/sentiment_reddit.csv"

if Path(REDDIT_SENT_PATH).exists():
    reddit_sent_raw = pd.read_csv(REDDIT_SENT_PATH)
    reddit_sent_raw["date"] = pd.to_datetime(reddit_sent_raw["date"], utc=True).dt.normalize().dt.tz_localize(None)
    reddit_sent_raw = reddit_sent_raw[
        (reddit_sent_raw["date"] >= DATE_MIN) & (reddit_sent_raw["date"] <= DATE_MAX)
    ]

    frames = []
    for group, prefix in [("TrumpBuzz", "reddit_trump"), ("HarrisBuzz", "reddit_harris"), ("ElectionBuzz", "reddit_election")]:
        sub = reddit_sent_raw[reddit_sent_raw["candidate"] == group].groupby("date").agg(
            sent_avg = ("vader_compound", "mean"),
            sent_std = ("vader_compound", "std"),
            **{e: (e, "mean") for e in EMOTION_COLS}
        )
        sub.columns = [f"{prefix}_{c}" for c in sub.columns]
        frames.append(sub)

    reddit_sentiment = pd.concat(frames, axis=1).reset_index()
    reddit_sentiment["reddit_sentiment_gap"] = (
        reddit_sentiment["reddit_trump_sent_avg"] - reddit_sentiment["reddit_harris_sent_avg"]
    )
    print(f"Reddit sentiment: {len(reddit_sentiment)} rows, {len(reddit_sentiment.columns)} cols")
    print(reddit_sentiment.columns.tolist())
else:
    print("WARNING: sentiment_reddit.csv not found — run Descriptive/1_reddit/1.5_sentiment_analysis.ipynb first")
    reddit_sentiment = None

Reddit sentiment: 122 rows, 32 cols
['date', 'reddit_trump_sent_avg', 'reddit_trump_sent_std', 'reddit_trump_fear', 'reddit_trump_anger', 'reddit_trump_anticipation', 'reddit_trump_trust', 'reddit_trump_surprise', 'reddit_trump_sadness', 'reddit_trump_disgust', 'reddit_trump_joy', 'reddit_harris_sent_avg', 'reddit_harris_sent_std', 'reddit_harris_fear', 'reddit_harris_anger', 'reddit_harris_anticipation', 'reddit_harris_trust', 'reddit_harris_surprise', 'reddit_harris_sadness', 'reddit_harris_disgust', 'reddit_harris_joy', 'reddit_election_sent_avg', 'reddit_election_sent_std', 'reddit_election_fear', 'reddit_election_anger', 'reddit_election_anticipation', 'reddit_election_trust', 'reddit_election_surprise', 'reddit_election_sadness', 'reddit_election_disgust', 'reddit_election_joy', 'reddit_sentiment_gap']


In [15]:
reddit_sentiment.head()

,date,reddit_trump_sent_avg,reddit_trump_sent_std,reddit_trump_fear,reddit_trump_anger,reddit_trump_anticipation,reddit_trump_trust,reddit_trump_surprise,reddit_trump_sadness,reddit_trump_disgust,...,reddit_election_sent_std,reddit_election_fear,reddit_election_anger,reddit_election_anticipation,reddit_election_trust,reddit_election_surprise,reddit_election_sadness,reddit_election_disgust,reddit_election_joy,reddit_sentiment_gap
0,2024-07-05,0.025650,0.573659,0.068965,0.066180,0.069694,0.104577,0.223423,0.061287,0.039978,...,0.550397,0.065275,0.067881,0.078509,0.128648,0.078366,0.069177,0.034990,0.044384,-0.063879
1,2024-07-06,0.012819,0.568414,0.066776,0.065132,0.069885,0.096334,0.244749,0.062865,0.040743,...,0.545136,0.065662,0.067147,0.078980,0.126347,0.086745,0.067717,0.034232,0.043639,-0.078085
2,2024-07-07,0.016148,0.569840,0.068851,0.064841,0.068682,0.100744,0.226159,0.065344,0.040839,...,0.556121,0.066048,0.069550,0.078713,0.130101,0.081038,0.067653,0.033216,0.044646,-0.087317
3,2024-07-08,0.034169,0.578541,0.067262,0.066298,0.068715,0.096905,0.224332,0.068851,0.039451,...,0.548962,0.062974,0.067690,0.077932,0.128898,0.080981,0.067520,0.033047,0.044764,-0.078122
4,2024-07-09,0.009348,0.574181,0.068972,0.065782,0.068875,0.096259,0.227549,0.071026,0.041589,...,0.556498,0.065287,0.069778,0.080897,0.124401,0.085933,0.071021,0.034760,0.044654,-0.090364


---
## 5. Newspapers

In [ ]:
# ─── Newspapers: article counts by leaning + buzz classification ───
# Uses the same buzz labeler as Bluesky & Reddit → candidate: TrumpBuzz / HarrisBuzz / ElectionBuzz
news_raw = pd.read_csv(
    "../data/2_silver/newspapers/mediacloud_articles_clean.csv",
    usecols=["date", "leaning", "title_clean"],
)
news_raw["date"] = pd.to_datetime(news_raw["date"], utc=True).dt.normalize().dt.tz_localize(None)
news_raw = news_raw[(news_raw["date"] >= DATE_MIN) & (news_raw["date"] <= DATE_MAX)]
news_raw = add_buzz_labels(news_raw, text_col="title_clean")

news_vol = news_raw.groupby("date").agg(
    news_total          = ("title_clean", "count"),
    news_trump_count    = ("candidate",   lambda x: (x == "TrumpBuzz").sum()),
    news_harris_count   = ("candidate",   lambda x: (x == "HarrisBuzz").sum()),
    news_election_count = ("candidate",   lambda x: (x == "ElectionBuzz").sum()),
    news_rep_count      = ("leaning",     lambda x: (x == "Republican").sum()),
    news_dem_count      = ("leaning",     lambda x: (x == "Democratic").sum()),
    news_cen_count      = ("leaning",     lambda x: (x == "Center/Unknown").sum()),
).reset_index()

news_vol["news_trump_share"]         = news_vol["news_trump_count"]  / news_vol["news_total"]
news_vol["news_harris_share"]        = news_vol["news_harris_count"] / news_vol["news_total"]
news_vol["news_attention_asymmetry"] = (
    (news_vol["news_trump_count"] - news_vol["news_harris_count"]) / news_vol["news_total"]
)

print(f"Newspaper volume: {len(news_vol)} rows, {len(news_vol.columns)} cols")
print(news_vol.columns.tolist())

In [ ]:
# ─── Newspapers: VADER + NRCLex per leaning — already daily in Silver ───
# Columns: vader_compound_mean/std/pos_neg_share × _dem/_rep/_cen
#        + nrc_<emotion> × _dem/_rep/_cen  (36 feature cols total)
news_sent = pd.read_csv("../data/2_silver/newspapers/sentiment_features_newspapers.csv")
news_sent["date"] = pd.to_datetime(news_sent["date"], utc=True).dt.normalize().dt.tz_localize(None)
news_sent = news_sent[(news_sent["date"] >= DATE_MIN) & (news_sent["date"] <= DATE_MAX)]

# Derived cross-leaning gap (republican minus democratic VADER compound)
if "vader_compound_mean_rep" in news_sent.columns and "vader_compound_mean_dem" in news_sent.columns:
    news_sent["news_vader_leaning_gap"] = (
        news_sent["vader_compound_mean_rep"] - news_sent["vader_compound_mean_dem"]
    )

print(f"Newspaper sentiment: {len(news_sent)} rows, {len(news_sent.columns)} cols")
print(news_sent.columns.tolist())

---
## 6. Google Trends

In [ ]:
# ─── Google Trends: daily search interest for 12 topic keywords ───
trends = pd.read_csv("../data/1_bronze/google_trends/trends_daily_stitched.csv")
trends["date"] = pd.to_datetime(trends["date"], utc=True).dt.normalize().dt.tz_localize(None)
trends = trends[(trends["date"] >= DATE_MIN) & (trends["date"] <= DATE_MAX)]

# Rename all keyword cols to gt_ prefix with underscored lowercase
rename = {c: "gt_" + c.lower().replace(" ", "_") for c in trends.columns if c != "date"}
trends = trends.rename(columns=rename)

print(f"Google Trends: {len(trends)} rows, {len(trends.columns)} cols")
print(trends.columns.tolist())

---
## 7. Polls

In [ ]:
# ─── Polls: daily aggregated poll numbers + 7-day rolling averages ───
polls_raw = pd.read_csv("../data/1_bronze/polls/wikipedia_polls.csv")
polls_raw["Date"] = polls_raw["Date"].astype(str).str.replace(r"\[.*?\]", "", regex=True).str.strip()
polls_raw["date"] = pd.to_datetime(polls_raw["Date"], errors="coerce")
polls_raw = polls_raw.dropna(subset=["date", "Trump", "Harris"])
polls_raw = polls_raw[(polls_raw["date"] >= DATE_MIN) & (polls_raw["date"] <= DATE_MAX)]

polls_daily = polls_raw.groupby("date").agg(
    poll_trump_avg  = ("Trump",  "mean"),
    poll_harris_avg = ("Harris", "mean"),
    poll_margin     = ("Margin", "mean"),
    poll_n_today    = ("Trump",  "count"),
).reset_index()

# Align to master date index, forward-fill days without polls
polls_daily = (
    pd.DataFrame({"date": DATES})
    .merge(polls_daily, on="date", how="left")
)
for col in ["poll_trump_avg", "poll_harris_avg", "poll_margin"]:
    polls_daily[col] = polls_daily[col].ffill()
polls_daily["poll_n_today"] = polls_daily["poll_n_today"].fillna(0).astype(int)

polls_daily["poll_trump_7d_avg"]  = polls_daily["poll_trump_avg"].rolling(7, min_periods=1).mean()
polls_daily["poll_harris_7d_avg"] = polls_daily["poll_harris_avg"].rolling(7, min_periods=1).mean()
polls_daily["poll_margin_7d_avg"] = polls_daily["poll_margin"].rolling(7, min_periods=1).mean()

print(f"Polls: {len(polls_daily)} rows, {len(polls_daily.columns)} cols")
print(polls_daily.columns.tolist())

---
## 8. Financials

In [ ]:
# ─── Financials: S&P500, Oil, VIX, bonds, USD + derived returns ───
market = (
    pd.read_csv("../data/1_bronze/financials/market.csv", parse_dates=["Date"])
    .rename(columns={"Date": "date", "SP500": "sp500", "Oil": "oil",
                     "VIX": "vix", "TenYearBond": "bond_10y", "USDIndex": "usd_index"})
    .sort_values("date")
)

market["sp500_return_1d"] = market["sp500"].pct_change(1)
market["sp500_return_7d"] = market["sp500"].pct_change(7)
market["sp500_vol_7d"]    = market["sp500_return_1d"].rolling(7).std()
market["oil_return_1d"]   = market["oil"].pct_change(1)
market["vix_change_1d"]   = market["vix"].diff(1)

market = market[(market["date"] >= DATE_MIN) & (market["date"] <= DATE_MAX)].reset_index(drop=True)

print(f"Financials: {len(market)} rows, {len(market.columns)} cols")
print(market.columns.tolist())
print(f"Note: market data only covers {market['date'].min().date()} → {market['date'].max().date()}")

---
## 9. Merge

In [ ]:
# ─── Merge all feature tables into basetable on date ───
sources = [
    (time_features,  "Time dimension"),
    (bsky_vol,       "Bluesky volume"),
    (bsky_sentiment, "Bluesky sentiment"),
    (reddit_vol,     "Reddit volume"),
    (news_vol,       "Newspaper volume"),
    (news_sent,      "Newspaper sentiment"),
    (trends,         "Google Trends"),
    (polls_daily,    "Polls"),
    (market,         "Financials"),
]
if reddit_sentiment is not None:
    sources.insert(4, (reddit_sentiment, "Reddit sentiment"))

for df_src, name in sources:
    basetable = basetable.merge(df_src, on="date", how="left")
    print(f"  + {name:<25}  {len(df_src.columns)-1:>3} cols  →  total {len(basetable.columns)}")

print(f"\nShape       : {basetable.shape}")
print(f"Date range  : {basetable['date'].min().date()} → {basetable['date'].max().date()}")
print(f"Missing (%):\n{basetable.isna().mean().sort_values(ascending=False).head(10).mul(100).round(1).to_string()}")

---
## 10. Lag features

In [ ]:
# ─── Lag features ───
basetable = basetable.sort_values("date").reset_index(drop=True)

# Polymarket level lags for AR models (lag1 already added in Section 1)
for k in range(2, 6):
    basetable[f"polymarket_trump_prob_lag{k}"] = basetable["polymarket_trump_prob"].shift(k)

# Daily price change + its lags
basetable["price_change"] = basetable["polymarket_trump_prob"].diff()
for k in range(1, 4):
    basetable[f"price_change_lag{k}"] = basetable["price_change"].shift(k)

# Key signal lags (t-1)
lag1_cols = [
    "bsky_sentiment_gap",
    "news_attention_asymmetry",
    "news_vader_leaning_gap",
    "poll_margin",
    "gt_trump",
    "gt_kamala",
    "vix",
    "sp500_return_1d",
]
if reddit_sentiment is not None:
    lag1_cols.append("reddit_sentiment_gap")

for col in lag1_cols:
    if col in basetable.columns:
        basetable[f"{col}_lag1"] = basetable[col].shift(1)

n_lag_cols = len([c for c in basetable.columns if "lag" in c or c == "price_change"])
print(f"Lag features added: {n_lag_cols} cols")
print(f"Final shape: {basetable.shape}")

---
## 11. Save

In [ ]:
# ─── Save final basetable ───
OUT = "../data/3_gold/basetable_ultimate.csv"
basetable.to_csv(OUT, index=False)

print(f"Saved → {OUT}")
print(f"Shape : {basetable.shape}  ({basetable.shape[1]-1} features + date)\n")

groups = [
    ("Target",          [c for c in basetable.columns if "polymarket" in c]),
    ("Time",            [c for c in basetable.columns if c in ["days_to_election", "is_weekend", "weekday", "campaign_week"]]),
    ("Bluesky",         [c for c in basetable.columns if c.startswith("bsky_")]),
    ("Reddit",          [c for c in basetable.columns if c.startswith("reddit_")]),
    ("Newspapers vol",  [c for c in basetable.columns if c.startswith("news_")]),
    ("Newspapers sent", [c for c in basetable.columns if c.startswith("vader_") or c.startswith("nrc_")]),
    ("Google Trends",   [c for c in basetable.columns if c.startswith("gt_")]),
    ("Polls",           [c for c in basetable.columns if c.startswith("poll_")]),
    ("Financials",      [c for c in basetable.columns if c in ["sp500", "oil", "vix", "bond_10y", "usd_index"]
                         or any(s in c for s in ["return", "vol_", "change_1d"])]),
    ("Lags",            [c for c in basetable.columns if "lag" in c or c == "price_change"]),
]

print(f"{'Group':<20} {'N cols':>6}")
print("-" * 28)
for name, cols in groups:
    print(f"{name:<20} {len(cols):>6}")
print("-" * 28)
print(f"{'TOTAL':<20} {sum(len(c) for _, c in groups):>6}")